### 🔧 Feature Engineering for Customer Churn Prediction
#### **Goal:** Transform raw data into model-ready features
#### **Techniques:** Encoding, scaling, feature creation, handling imbalance

##### 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Professional settings
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-darkgrid')

print("✅ Libraries imported")

✅ Libraries imported


#### 2. Load Cleaned Data from EDA
#### Load the data we saved during EDA

In [6]:
df = pd.read_csv('C:/Users/pc\Documents/customer-churn-enterprise/data/processed\churn_data_clean.csv')

print(f"✅ Data loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

✅ Data loaded: 7043 rows, 21 columns


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### 3. Separate Features and Target
#### Make a copy to avoid modifying original

In [7]:
data = df.copy()

# Separate features and target
X = data.drop('Churn', axis=1)
y = data['Churn']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nTarget distribution:")
print(y.value_counts())
print(f"Churn Rate: {(y == 'Yes').mean() * 100:.2f}%")

# %% [markdown]
# ## 4. Handle Categorical Variables

# %%
# Identify categorical and numerical columns
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Remove customerID from features (it's just an identifier)
if 'customerID' in categorical_cols:
    categorical_cols.remove('customerID')
    X = X.drop('customerID', axis=1)

print(f"📊 Categorical columns: {len(categorical_cols)}")
print(categorical_cols)
print(f"\n📊 Numerical columns: {len(numerical_cols)}")
print(numerical_cols)

Features shape: (7043, 20)
Target shape: (7043,)

Target distribution:
Churn
No     5174
Yes    1869
Name: count, dtype: int64
Churn Rate: 26.54%
📊 Categorical columns: 16
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TotalCharges']

📊 Numerical columns: 3
['SeniorCitizen', 'tenure', 'MonthlyCharges']


#### 4.1 One-Hot Encoding for Categorical Variables
#### Perform one-hot encoding

In [8]:
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

print(f"✅ After encoding: {X_encoded.shape[1]} features")
print(f"Original features: {X.shape[1]}")
print(f"New features created: {X_encoded.shape[1] - X.shape[1]}")

# Display first few encoded columns
encoded_cols = [col for col in X_encoded.columns if any(cat in col for cat in categorical_cols)]
print("\nSample of encoded columns:")
X_encoded[encoded_cols[:5]].head()

# %% [markdown]
# ### 4.2 Convert Target Variable to Binary

# %%
# Convert Churn to binary (Yes=1, No=0)
y_binary = (y == 'Yes').astype(int)

print("Target encoding:")
print(f"No Churn (0): {(y_binary == 0).sum()} customers")
print(f"Churn (1): {(y_binary == 1).sum()} customers")

✅ After encoding: 6559 features
Original features: 19
New features created: 6540

Sample of encoded columns:
Target encoding:
No Churn (0): 5174 customers
Churn (1): 1869 customers


#### 4.2 Convert Target Variable to Binary
#### Convert Churn to binary (Yes=1, No=0)

In [9]:
y_binary = (y == 'Yes').astype(int)

print("Target encoding:")
print(f"No Churn (0): {(y_binary == 0).sum()} customers")
print(f"Churn (1): {(y_binary == 1).sum()} customers")

Target encoding:
No Churn (0): 5174 customers
Churn (1): 1869 customers


#### 5. Feature Scaling
#### Identify numerical columns in the encoded dataset

In [10]:
numerical_cols_encoded = [col for col in numerical_cols if col in X_encoded.columns]

print(f"Scaling {len(numerical_cols_encoded)} numerical features:")
print(numerical_cols_encoded)

# Initialize scaler
scaler = StandardScaler()

# Fit and transform numerical features
X_encoded[numerical_cols_encoded] = scaler.fit_transform(X_encoded[numerical_cols_encoded])

# Display scaled values
print("\n✅ After scaling (first 5 rows):")
X_encoded[numerical_cols_encoded].head()


Scaling 3 numerical features:
['SeniorCitizen', 'tenure', 'MonthlyCharges']

✅ After scaling (first 5 rows):


,SeniorCitizen,tenure,MonthlyCharges
0,-0.439916,-1.277445,-1.160323
1,-0.439916,0.066327,-0.259629
2,-0.439916,-1.236724,-0.362660
3,-0.439916,0.514251,-0.746535
4,-0.439916,-1.236724,0.197365


#### 6. Create New Features (Feature Engineering)
#### Add engineered features to enhance model performance
##### 6.1 Average revenue per month of tenure

In [12]:
X['TotalCharges'] = pd.to_numeric(X['TotalCharges'], errors='coerce')
X['TotalCharges'] = X['TotalCharges'].fillna(0)
X_encoded['Avg_Revenue_Per_Month'] = X['TotalCharges'] / (X['tenure'] + 1)

# 6.2 Tenure groups (categorize customers by loyalty)
X_encoded['Tenure_Group'] = pd.cut(X['tenure'], 
                                    bins=[-1, 6, 12, 24, 48, 100], 
                                    labels=['0-6 Months', '6-12 Months', '1-2 Years', '2-4 Years', '4+ Years'])

# One-hot encode tenure groups
tenure_dummies = pd.get_dummies(X_encoded['Tenure_Group'], prefix='Tenure')
X_encoded = pd.concat([X_encoded, tenure_dummies], axis=1)
X_encoded = X_encoded.drop('Tenure_Group', axis=1)

# 6.3 Service count (how many services customer has)
service_columns = ['PhoneService', 'MultipleLines', 'InternetService',
                   'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                   'TechSupport', 'StreamingTV', 'StreamingMovies']

# Count services (simplified approach)
service_count = 0
for col in service_columns:
    if col in X.columns:
        if X[col].dtype == 'object':
            service_count += (X[col] != 'No').astype(int)
        else:
            service_count += X[col]

X_encoded['Total_Services'] = service_count

# 6.4 Family indicator (Partner + Dependents)
if 'Partner' in X.columns and 'Dependents' in X.columns:
    X_encoded['Has_Family'] = ((X['Partner'] == 'Yes') | (X['Dependents'] == 'Yes')).astype(int)

# 6.5 High value customer (monthly charges above median)
monthly_charges_median = X['MonthlyCharges'].median()
X_encoded['High_Value'] = (X['MonthlyCharges'] > monthly_charges_median).astype(int)

print(f"✅ Added engineered features. New shape: {X_encoded.shape}")

✅ Added engineered features. New shape: (7043, 6568)


#### 7. Train-Test Split
#### Split data into training and testing sets

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_binary, test_size=0.2, random_state=42, stratify=y_binary
)

print(f"📊 Training set: {X_train.shape[0]} samples")
print(f"📊 Test set: {X_test.shape[0]} samples")
print(f"\nTraining set churn distribution:")
print(y_train.value_counts(normalize=True))
print(f"\nTest set churn distribution:")
print(y_test.value_counts(normalize=True))

📊 Training set: 5634 samples
📊 Test set: 1409 samples

Training set churn distribution:
Churn
0    0.734647
1    0.265353
Name: proportion, dtype: float64

Test set churn distribution:
Churn
0    0.734564
1    0.265436
Name: proportion, dtype: float64


#### 8. Handle Imbalanced Data with SMOTE

In [14]:
from collections import Counter

print(f"Original training set shape: {Counter(y_train)}")

# Apply SMOTE
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f"Resampled training set shape: {Counter(y_train_resampled)}")
print(f"✅ SMOTE applied successfully")


Original training set shape: Counter({0: 4139, 1: 1495})
Resampled training set shape: Counter({0: 4139, 1: 4139})
✅ SMOTE applied successfully


#### 9. Save Processed Data
#### Save all processed datasets

In [15]:
X_train.to_csv('C:/Users/pc\Documents/customer-churn-enterprise/data/processed/X_train.csv', index=False)
X_test.to_csv('C:/Users/pc\Documents/customer-churn-enterprise/data/processed/X_test.csv', index=False)
y_train.to_csv('C:/Users/pc\Documents/customer-churn-enterprise/data/processed/y_train.csv', index=False)
y_test.to_csv('C:/Users/pc\Documents/customer-churn-enterprise/data/processed/y_test.csv', index=False)

# Save resampled data for models that need it
X_train_resampled.to_csv('C:/Users/pc\Documents/customer-churn-enterprise/data/processed/X_train_resampled.csv', index=False)
pd.Series(y_train_resampled).to_csv('C:/Users/pc\Documents/customer-churn-enterprise/data/processed/y_train_resampled.csv', index=False)

# Save feature names for later use
feature_names = X_train.columns.tolist()
pd.Series(feature_names).to_csv('C:/Users/pc\Documents/customer-churn-enterprise/data/processed/feature_names.csv', index=False)

# Save scaler for later use in production
import joblib
joblib.dump(scaler, 'C:/Users/pc\Documents/customer-churn-enterprise/models/scaler.pkl')

print("\n✅ All processed data saved to data/processed/")
print(f"Features created: {len(feature_names)}")


✅ All processed data saved to data/processed/
Features created: 6568


#### 10. Feature Engineering Summary

In [16]:
print("\n" + "="*80)
print("📋 FEATURE ENGINEERING SUMMARY")
print("="*80)
print(f"""
Original features: {X.shape[1]}
After encoding: {X_encoded.shape[1]}
New engineered features: {X_encoded.shape[1] - X.shape[1]}

🔹 ENCODING TECHNIQUES:
- One-Hot Encoding for categorical variables
- Binary encoding for target variable
- StandardScaler for numerical features

🔹 NEW FEATURES CREATED:
1. Avg_Revenue_Per_Month: Total charges per month of tenure
2. Tenure groups: Customer loyalty categories
3. Total_Services: Number of services subscribed
4. Has_Family: Whether customer has family
5. High_Value: Customer with above-median monthly charges

🔹 DATA SPLIT:
- Training: {X_train.shape[0]} samples ({X_train.shape[0]/len(X_encoded)*100:.1f}%)
- Test: {X_test.shape[0]} samples ({X_test.shape[0]/len(X_encoded)*100:.1f}%)

🔹 IMBALANCE HANDLING:
- Original: {Counter(y_train)}
- After SMOTE: {Counter(y_train_resampled)}
""")

print("\n✅ Feature Engineering Complete! Ready for Model Training")


📋 FEATURE ENGINEERING SUMMARY

Original features: 19
After encoding: 6568
New engineered features: 6549

🔹 ENCODING TECHNIQUES:
- One-Hot Encoding for categorical variables
- Binary encoding for target variable
- StandardScaler for numerical features

🔹 NEW FEATURES CREATED:
1. Avg_Revenue_Per_Month: Total charges per month of tenure
2. Tenure groups: Customer loyalty categories
3. Total_Services: Number of services subscribed
4. Has_Family: Whether customer has family
5. High_Value: Customer with above-median monthly charges

🔹 DATA SPLIT:
- Training: 5634 samples (80.0%)
- Test: 1409 samples (20.0%)

🔹 IMBALANCE HANDLING:
- Original: Counter({0: 4139, 1: 1495})
- After SMOTE: Counter({0: 4139, 1: 4139})


✅ Feature Engineering Complete! Ready for Model Training
